## Process Dead-Letter Queue (DLQ) Messages

### Installing Libraries and Utilities

In [ ]:
%pip install azure-servicebus==7.14.3 openai==2.38.0 python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# loading service bus configurations
service_bus_connection_string = os.getenv("SERVICE_BUS_CONNECTION_STRING")
service_bus_queue_name = os.getenv("SERVICE_BUS_QUEUE_NAME")

# loading azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
chat_completions_model = os.getenv("CHAT_COMPLETIONS_MODEL")

### Creating the Service Bus Client

In [ ]:
from azure.servicebus import ServiceBusClient

sb_client = ServiceBusClient.from_connection_string(
    conn_str = service_bus_connection_string
)

### Helper Function to Process User Queries

In [ ]:
from openai import AzureOpenAI

def process_user_message(user_query, llm):
    azure_openai_client = AzureOpenAI(
        azure_endpoint = azure_openai_endpoint,
        api_version = "2024-06-01",
        api_key = azure_openai_api_key
    )

    response = azure_openai_client.chat.completions.create(
        model = llm,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful AI assistant"
            },
            {
                "role": "user",
                "content": user_query
            }
        ],
        temperature = 0.7
    )

    return response.choices[0].message.content

### Process DLQ Messages Reliably

In [ ]:
from azure.servicebus import ServiceBusSubQueue
import json

# creating the DLQ receiver object
dlq_receiver = sb_client.get_queue_receiver(
    queue_name = service_bus_queue_name,
    sub_queue = ServiceBusSubQueue.DEAD_LETTER,
    max_wait_time = 10
)

for msg in dlq_receiver:
    # extracting the request payload
    payload = json.loads(str(msg))

    # processing request with the correct model name
    print("user query: {}".format(payload.get("prompt")))
    print("\n")
    assistant_response = process_user_message(payload.get("prompt"), chat_completions_model)
    print("Assistant Reponse: {}".format(assistant_response))
    print("============================================")
    
    # marking the payload as completed after successful execution
    dlq_receiver.complete_message(msg)
    